# **Evidencia de Aprendizaje 3**  
## Transformación y Análisis de Datos Enriquecidos

**Autores:**  
Katerin Lopez Moros  
Bayron Meza Guzman

Programación para análisis de datos  
Ana Lopez


*Nota: Este es el archivo main (principal) en el que se enlazan los demás*

## **Scraper**

El desarrollo e implementación del scraper lo conforma la configuración del entorno (config.py), la extracción de los datos (extraccion.py) y el procesamiento que es la limpieza y transformación de los datos (procesamiento.py) lo cual ejecutamos aquí en el main para paso a paso.

In [21]:
# ── 1: Configuración del entorno ──────────────────────────────
from config import configurar_driver

driver = configurar_driver(headless=True)

Driver configurado correctamente.


In [22]:
# ── 2: Limpiamos la Base de Datos ──────────────────────────────
from limpiar_bd import limpiar_base_datos

limpiar_base_datos()


LIMPIANDO BASE DE DATOS
 Tabla 'precios' limpia
 Tabla 'productos' limpia
 Tabla 'tiendas' limpia
 Tabla 'indicadores_economicos' limpia

 Base de datos lista para recibir datos

 Estado de la BD tras limpieza

precios                         vacía
productos                       vacía
tiendas                         vacía
indicadores_economicos          vacía


In [23]:
# ── 3: Extracción de Datos ──────────────────────────────

import time
import os
from config import configurar_driver
from extraccion import (
    obtener_productos,
    guardar_csv,
    guardar_json
)

# =========================================================
# URLS
# =========================================================

URLS = [

    "https://www.exito.com/tecnologia",
    "https://www.exito.com/mercado",
    "https://www.exito.com/hogar",
    "https://www.exito.com/bebes",
    "https://www.exito.com/cuidado-personal",
    "https://www.exito.com/belleza",
    "https://www.exito.com/celulares"

]

# =========================================================
# SCRAPING
# =========================================================

raw = []

try:

    total_categorias = len(URLS)

    for i, url in enumerate(URLS, start=1):

        print("\n" + "=" * 70)
        print(f"CATEGORÍA {i}/{total_categorias}")
        print(url)
        print("=" * 70)

        try:

            productos = obtener_productos(driver, url)
            raw.extend(productos)

            print(

                f"\nProductos obtenidos "
                f"de categoría: {len(productos)}"

            )

            print(

                f"Total global acumulado: "
                f"{len(raw)}"

            )

        except Exception as e:

            print(f"\nError en categoría: {e}")

        time.sleep(3)

except Exception as e:

    print(f"\nERROR GENERAL: {e}")

finally:

    driver.quit()
    print("\nDriver cerrado correctamente")

# =========================================================
# RESUMEN FINAL
# =========================================================

print("\n" + "=" * 70)
print("SCRAPING FINALIZADO")
print("=" * 70)
print(f"\nTOTAL FINAL: {len(raw)}")

# =========================================================
# GUARDAR ARCHIVOS
# =========================================================

print("\nGuardando CSV...")
guardar_csv(raw)
print("Guardando JSON...")
guardar_json(raw)
print("\nArchivos creados correctamente")

# =========================================================
# MOSTRAR RUTA DE GUARDADO
# =========================================================

print("\nCarpeta actual:")
print(os.getcwd())

# =========================================================
# VERIFICAR ARCHIVOS
# =========================================================

print("\nArchivos disponibles:")

for archivo in os.listdir():

    if archivo.endswith(".csv") or archivo.endswith(".json"):

        print(f" - {archivo}")



CATEGORÍA 1/7
https://www.exito.com/tecnologia

CATEGORÍA: TECNOLOGIA
  Scroll   0 →  18 nuevos | total: 18
  Scroll   1 → sin nuevos (2/5)
  Scroll   2 → sin nuevos (4/5)
  Scroll   3 → sin nuevos (6/5)
  Fin de página detectado.
  Total extraídos: 18

Productos obtenidos de categoría: 18
Total global acumulado: 18

CATEGORÍA 2/7
https://www.exito.com/mercado

CATEGORÍA: MERCADO
  Scroll   0 →  17 nuevos | total: 17
  Scroll   1 → sin nuevos (1/5)
  Scroll   2 → sin nuevos (3/5)
  Scroll   3 → sin nuevos (5/5)
  Fin de página detectado.
  Total extraídos: 17

Productos obtenidos de categoría: 17
Total global acumulado: 35

CATEGORÍA 3/7
https://www.exito.com/hogar

CATEGORÍA: HOGAR
  Scroll   0 →  16 nuevos | total: 16
  Scroll   1 → sin nuevos (2/5)
  Scroll   2 → sin nuevos (4/5)
  Scroll   3 → sin nuevos (6/5)
  Fin de página detectado.
  Total extraídos: 16

Productos obtenidos de categoría: 16
Total global acumulado: 51

CATEGORÍA 4/7
https://www.exito.com/bebes

CATEGORÍA: BEBE

In [24]:
# ── 4: Procesamiento ───────────────────────────────────────────
from procesamiento import procesar_todo

tablas = procesar_todo(raw)


 ___________Procesamiento de datos___________
  productos: 116 filas
  tiendas: 1 fila(s)
  precios: 1 outliers detectados (conservados)
  precios: 97 filas
  indicadores_economicos: tabla vacía


## **Ingesta a la BD**

Aquí se enlaza el archivo de la ingesta de datos (ingesta.py) que conecta con la base de datos y hace la inserción de los datos a la misma

In [25]:
# ── 5A: Validación previa a la ingesta ───────────────────────────
print('Validando tablas antes de la ingesta:')
for nombre, df in tablas.items():
    print(f'  {nombre}: {len(df)} filas, columnas={list(df.columns)}')
print('')
print('Si las tablas son correctas, ejecuta la siguiente celda de ingesta.')


Validando tablas antes de la ingesta:
  productos: 116 filas, columnas=['id_producto', 'codigo_barras', 'nombre', 'categoria', 'marca', 'contenido_neto', 'unidad']
  tiendas: 1 filas, columnas=['id_tienda', 'nombre', 'tipo_canal', 'ciudad']
  precios: 97 filas, columnas=['id_producto', 'id_tienda', 'precio_venta', 'precio_original', 'fecha_registro', 'fuente']
  indicadores_economicos: 0 filas, columnas=['Fecha']

Si las tablas son correctas, ejecuta la siguiente celda de ingesta.


In [26]:
# ── 5B: Ingesta en la base de datos ────────────────────────────
from ingesta import ingestar

ingestar(tablas)


___________Ingesta a la base de datos___________
  Conexion establecida: consumo_masivo.db
  Tablas verificadas/creadas correctamente.
  tiendas                             1 registros totales en BD
  productos                         116 registros totales en BD
  precios                            90 registros totales en BD
  indicadores_economicos: vacia, se omite.

  Pipeline de ingesta completado.


## **Pruebas y Validaciones**

Aquí realizamos pruebas para verificar que el scraper funciona correctamente y que los datos se almacenan adecuadamente en la base de datos

In [27]:
# ── 6: Pruebas de verificación ────────────────────────────────

# Importamos librerias necesarias
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///consumo_masivo.db')

# Prueba 1: Las 4 tablas existen y tienen registros
print('── Prueba 1: Conteo de registros por tabla ──')
for tabla in ['productos', 'tiendas', 'precios', 'indicadores_economicos']:
    with engine.connect() as conn:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {tabla}')).scalar()
    estado = 'OK' if n > 0 or tabla == 'indicadores_economicos' else 'REVISAR'
    print(f'  {estado} {tabla:<30} {n:>6,} registros')


# Prueba 2: No hay duplicados en productos
print('\n── Prueba 2: Duplicados en productos ──')
with engine.connect() as conn:
    duplicados = conn.execute(text(
        'SELECT COUNT(*) FROM (SELECT codigo_barras FROM productos GROUP BY codigo_barras HAVING COUNT(*) > 1)'
    )).scalar()
print(f'  {"OK" if duplicados == 0 else "REVISAR"} Duplicados por codigo_barras: {duplicados}')


# Prueba 3: Todos los precios tienen id_producto válido (integridad referencial)
print('\n── Prueba 3: Integridad referencial precios → productos ──')
with engine.connect() as conn:
    huerfanos = conn.execute(text(
        'SELECT COUNT(*) FROM precios WHERE id_producto NOT IN (SELECT id_producto FROM productos)'
    )).scalar()
print(f'  {"OK" if huerfanos == 0 else "REVISAR"} Precios sin producto asociado: {huerfanos}')


# Prueba 4: Precio de venta siempre > 0
print('\n── Prueba 4: Precios de venta válidos ──')
with engine.connect() as conn:
    invalidos = conn.execute(text(
        'SELECT COUNT(*) FROM precios WHERE precio_venta <= 0 OR precio_venta IS NULL'
    )).scalar()
print(f'  {"OK" if invalidos == 0 else "REVISAR"} Precios inválidos: {invalidos}')


# Prueba 5: Muestra de los primeros 5 productos almacenados
print('\n── Prueba 5: Muestra de productos en BD ──')
with engine.connect() as conn:
    df_muestra = pd.read_sql('SELECT nombre, categoria, marca, precio_venta FROM productos p JOIN precios pr ON p.id_producto = pr.id_producto LIMIT 5', conn)
print(df_muestra.to_string(index=False))

── Prueba 1: Conteo de registros por tabla ──
  OK productos                         116 registros
  OK tiendas                             1 registros
  OK precios                            90 registros
  OK indicadores_economicos              0 registros

── Prueba 2: Duplicados en productos ──
  OK Duplicados por codigo_barras: 0

── Prueba 3: Integridad referencial precios → productos ──
  OK Precios sin producto asociado: 0

── Prueba 4: Precios de venta válidos ──
  OK Precios inválidos: 0

── Prueba 5: Muestra de productos en BD ──
                                                         nombre  categoria   marca  precio_venta
Parlante LG GRAB.ACOLLBK 20W Negro Sonido IA, Diseño ergonómico tecnologia      LG        399900
         Televisor KALLEY 60 pulgadas LED Uhd4K Smart TV 60G310 tecnologia  KALLEY       1401904
Televisor SAMSUNG 58 pulgadas LED Uhd4K Smart TV UN58U8000FKXZL tecnologia SAMSUNG       1705904
              Televisor TCL 50 pulgadas QLED Fhd Smart TV 50S5K te

---
## **Etapa 4 CRISP-DM — Transformación y Enriquecimiento de Datos**

Con los datos ya extraídos e ingestados en la BD, aplicamos una capa de enriquecimiento
que agrega 14 campos nuevos a partir de los datos crudos del scraper.
El módulo  define todas las transformaciones y  orquesta el proceso completo.


### ¿Qué enriquecemos y por qué?

| Campo nuevo | Origen | Utilidad en Power BI |
|---|---|---|
| precio_venta | Extraído del texto sucio del scraper | Eje numérico real para análisis |
| precio_original | Primer valor del campo precio (tachado) | Comparar contra precio de oferta |
| descuento_pct | Porcentaje explícito en tarjeta (-56%) | Filtrar/ordenar por descuento |
| ahorro_cop | precio_original menos precio_venta | Mostrar ahorro absoluto en COP |
| tiene_descuento | Bandera 0/1 | Segmentar productos en oferta |
| rango_precio | Umbrales por categoría | Segmentar mercado por nivel |
| subcategoria | Inferido del nombre por regex | Drill-down más granular |
| almacenamiento_gb | Extraído del nombre del producto | Análisis de especificaciones |
| ram_gb | Extraído del nombre del producto | Cruzar precio vs RAM |
| conectividad_5g | Detectado en nombre del producto | Comparar precios 5G vs 4G |
| marca_normalizada | Marca en Title Case | Consistencia en visualizaciones |
| nombre_limpio | Nombre sin caracteres extraños | Etiquetas limpias en gráficos |
| fecha_extraccion | Fecha de ejecución del pipeline | Trazabilidad temporal |
| fuente | Literal: exito.com | Auditoría del origen del dato |


In [28]:
# ── 7: Cargar datos crudos del scraper ──────────────────────────────
import json

ARCHIVO_JSON = "productos_exito.json"

with open(ARCHIVO_JSON, encoding="utf-8") as f:
    raw_scraper = json.load(f)

print(f"Registros crudos cargados: {len(raw_scraper)}")
print(f"Campos disponibles: {list(raw_scraper[0].keys())}")


Registros crudos cargados: 123
Campos disponibles: ['nombre', 'precio_venta', 'precio_original', 'descuento_pct', 'marca', 'categoria', 'enlace', 'imagen']


In [29]:
# ── 8: Aplicar enriquecimiento (modelo.py) ───────────────────────────
from modelo import enriquecer

datos_enriquecidos = enriquecer(raw_scraper)

print(f"Registros tras enriquecimiento: {len(datos_enriquecidos)}")
print(f"Campos nuevos disponibles: {list(datos_enriquecidos[0].keys())}")


Registros tras enriquecimiento: 123
Campos nuevos disponibles: ['nombre', 'marca', 'categoria', 'enlace', 'imagen', 'precio_venta', 'precio_original', 'descuento_pct', 'ahorro_cop', 'tiene_descuento', 'rango_precio', 'subcategoria', 'almacenamiento_gb', 'ram_gb', 'conectividad_5g', 'marca_normalizada', 'nombre_limpio', 'fecha_extraccion', 'fuente']


In [31]:
# ── 9: Vista previa del enriquecimiento ──────────────────────────────
import pandas as pd

df_enr = pd.DataFrame(datos_enriquecidos)

print("── Primeros 3 registros enriquecidos ──")
cols_preview = [
    "nombre_limpio", "marca_normalizada", "categoria", "subcategoria",
    "precio_venta", "precio_original", "descuento_pct", "ahorro_cop",
    "rango_precio", "conectividad_5g"
]
print(df_enr[cols_preview].head(3).to_string(index=False))

print("── Estadísticas generales ──")
print(f"  Con precio_venta:     {df_enr['precio_venta'].notna().sum()}")
print(f"  Con descuento:        {df_enr['tiene_descuento'].sum()}")
print(f"  Descuento promedio:   {df_enr['descuento_pct'].mean():.1f}%")
print(f"  Ahorro promedio:      ${df_enr['ahorro_cop'].mean():,.0f} COP")
print(f"  Con conectividad 5G:  {df_enr['conectividad_5g'].sum()}")

print("── Distribución por rango de precio ──")
print(df_enr["rango_precio"].value_counts().to_string())

print("── Top 10 subcategorías ──")
print(df_enr["subcategoria"].value_counts().head(10).to_string())


── Primeros 3 registros enriquecidos ──
                                                  nombre_limpio marca_normalizada  categoria subcategoria  precio_venta  precio_original  descuento_pct  ahorro_cop rango_precio  conectividad_5g
Parlante LG GRAB.ACOLLBK 20W Negro Sonido IA, Diseño ergonómico                Lg tecnologia        audio      399900.0         799900.0           50.0    400000.0        medio                0
         Televisor KALLEY 60 pulgadas LED Uhd4K Smart TV 60G310            Kalley tecnologia    televisor     1401904.0        3699900.0           62.0   2297996.0         alto                0
Televisor SAMSUNG 58 pulgadas LED Uhd4K Smart TV UN58U8000FKXZL           Samsung tecnologia    televisor     1705904.0        3599900.0           52.0   1893996.0         alto                0
── Estadísticas generales ──
  Con precio_venta:     97
  Con descuento:        81
  Descuento promedio:   47.1%
  Ahorro promedio:      $927,080 COP
  Con conectividad 5G:  10
── Dist

In [32]:
# ── 10: Guardar CSV y JSON enriquecidos ──────────────────────────────
import os
import pandas as pd

CSV_DIR = os.getcwd()
CSV_PATH = os.path.join(CSV_DIR, "productos_enriquecidos.csv")
JSON_PATH = os.path.join(CSV_DIR, "productos_enriquecidos.json")

COLUMNAS_CSV = [
    "nombre_limpio",
    "marca_normalizada",
    "categoria",
    "subcategoria",
    "precio_venta",
    "precio_original",
    "descuento_pct",
    "ahorro_cop",
    "tiene_descuento",
    "rango_precio",
    "almacenamiento_gb",
    "ram_gb",
    "conectividad_5g",
    "fecha_extraccion",
    "fuente",
    "enlace",
]

# Crear copia para exportar
df_export = df_enr.copy()

# Columnas numéricas enteras
columnas_enteras = [
    "precio_venta",
    "precio_original",
    "descuento_pct",
    "ahorro_cop",
    "almacenamiento_gb",
    "ram_gb",
    "tiene_descuento",
    "conectividad_5g",
]

for col in columnas_enteras:
    if col in df_export.columns:
        df_export[col] = df_export[col].apply(
            lambda x: "" if pd.isna(x) else str(int(float(x)))
        )

# Guardar CSV
df_export[COLUMNAS_CSV].to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Guardar JSON
df_enr.to_json(
    JSON_PATH,
    orient="records",
    force_ascii=False,
    indent=2
)

print(f"CSV guardado en:  {CSV_PATH}")
print(f"JSON guardado en: {JSON_PATH}")
print(f"Total registros:  {len(df_enr)}")

print("\nVerificación:")
print(
    df_export[
        [
            "precio_venta",
            "precio_original",
            "descuento_pct",
            "ahorro_cop",
        ]
    ].head()
)


CSV guardado en:  c:\Users\bayro\Downloads\Entrega 2 - Corregido\Entrega 2\Entrega 2\productos_enriquecidos.csv
JSON guardado en: c:\Users\bayro\Downloads\Entrega 2 - Corregido\Entrega 2\Entrega 2\productos_enriquecidos.json
Total registros:  123

Verificación:
  precio_venta precio_original descuento_pct ahorro_cop
0       399900          799900            50     400000
1      1401904         3699900            62    2297996
2      1705904         3599900            52    1893996
3       984905         2299900            57    1314995
4      1724905         3499900            50    1774995


In [33]:
# ── 11: Cargar datos enriquecidos a SQLite ────────────────────────────
from sqlalchemy import create_engine, text

DB_PATH   = "consumo_masivo.db"
TABLA_ENR = "productos_enriquecidos"

DDL = f"""
CREATE TABLE IF NOT EXISTS {TABLA_ENR} (
    nombre_limpio       TEXT NOT NULL,
    marca_normalizada   TEXT,
    categoria           TEXT,
    subcategoria        TEXT,
    precio_venta        REAL,
    precio_original     REAL,
    descuento_pct       INTEGER,
    ahorro_cop          REAL,
    tiene_descuento     INTEGER,
    rango_precio        TEXT,
    almacenamiento_gb   INTEGER,
    ram_gb              INTEGER,
    conectividad_5g     INTEGER,
    fecha_extraccion    TEXT,
    fuente              TEXT,
    enlace              TEXT,
    nombre              TEXT,
    PRIMARY KEY (nombre_limpio, categoria)
)
"""

engine_enr = create_engine(f"sqlite:///{DB_PATH}", echo=False)

with engine_enr.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {TABLA_ENR}"))
    conn.execute(text(DDL))

COLS_BD = [
    "nombre_limpio", "marca_normalizada", "categoria", "subcategoria",
    "precio_venta", "precio_original", "descuento_pct", "ahorro_cop",
    "tiene_descuento", "rango_precio",
    "almacenamiento_gb", "ram_gb", "conectividad_5g",
    "fecha_extraccion", "fuente", "enlace", "nombre",
]

df_bd = df_enr[COLS_BD].copy()
df_bd["tiene_descuento"] = df_bd["tiene_descuento"].astype(int)
df_bd["conectividad_5g"] = df_bd["conectividad_5g"].astype(int)
df_bd = df_bd.drop_duplicates(subset=["nombre_limpio", "categoria"])

filas   = [tuple(r) for r in df_bd.itertuples(index=False, name=None)]
cols    = ", ".join(COLS_BD)
holders = ", ".join(["?"] * len(COLS_BD))
sql_ins = f"INSERT OR IGNORE INTO {TABLA_ENR} ({cols}) VALUES ({holders})"

with engine_enr.begin() as conn:
    conn.exec_driver_sql(sql_ins, filas)

with engine_enr.connect() as conn:
    n = conn.execute(text(f"SELECT COUNT(*) FROM {TABLA_ENR}")).scalar()

print(f"Tabla '{TABLA_ENR}' creada con {n} registros.")


Tabla 'productos_enriquecidos' creada con 117 registros.


### Validaciones de calidad del enriquecimiento


In [34]:
# ── 12: Validaciones del enriquecimiento ─────────────────────────────

print("═" * 55)
print("  VALIDACIONES DE CALIDAD — DATOS ENRIQUECIDOS")
print("═" * 55)

errores = 0

# V1: Sin nulos en nombre_limpio
n1 = df_enr["nombre_limpio"].isna().sum()
print(f"  V1 nombre_limpio sin nulos       {'✓ PASS' if n1==0 else f'✗ FAIL ({n1} nulos)'}")
errores += n1 > 0

# V2: precio_venta > 0 cuando existe
n2 = (df_enr["precio_venta"].notna() & (df_enr["precio_venta"] <= 0)).sum()
print(f"  V2 precio_venta > 0              {'✓ PASS' if n2==0 else f'✗ FAIL ({n2})'}")
errores += n2 > 0

# V3: precio_original >= precio_venta
mask = df_enr["precio_venta"].notna() & df_enr["precio_original"].notna()
n3 = (df_enr[mask]["precio_original"] < df_enr[mask]["precio_venta"]).sum()
print(f"  V3 precio_original >= precio_venta {'✓ PASS' if n3==0 else f'✗ FAIL ({n3})'}")
errores += n3 > 0

# V4: Sin duplicados
n4 = df_enr.duplicated(subset=["nombre_limpio", "categoria"]).sum()
print(f"  V4 sin duplicados (nombre+cat)   {'✓ PASS' if n4==0 else f'✗ FAIL ({n4})'}")
errores += n4 > 0

# V5: rango_precio nunca vacío
n5 = (df_enr["rango_precio"] == "sin_precio").sum()
print(f"  V5 rango_precio completo         INFO ({n5} productos sin precio)")

# V6: descuento_pct entre 1 y 99
mask2 = df_enr["descuento_pct"].notna()
n6 = ((df_enr[mask2]["descuento_pct"] < 1) | (df_enr[mask2]["descuento_pct"] > 99)).sum()
print(f"  V6 descuento_pct en [1-99]       {'✓ PASS' if n6==0 else f'✗ FAIL ({n6})'}")
errores += n6 > 0

print(f" Resultado: {6 - errores}/6 validaciones pasadas.")


═══════════════════════════════════════════════════════
  VALIDACIONES DE CALIDAD — DATOS ENRIQUECIDOS
═══════════════════════════════════════════════════════
  V1 nombre_limpio sin nulos       ✓ PASS
  V2 precio_venta > 0              ✓ PASS
  V3 precio_original >= precio_venta ✓ PASS
  V4 sin duplicados (nombre+cat)   ✗ FAIL (6)
  V5 rango_precio completo         INFO (26 productos sin precio)
  V6 descuento_pct en [1-99]       ✓ PASS
 Resultado: 5/6 validaciones pasadas.


### Análisis exploratorio de los datos enriquecidos


In [35]:
# ── 13: Análisis exploratorio ────────────────────────────────────────

print("── Precio promedio por categoría ──")
resumen = df_enr.groupby("categoria")["precio_venta"].agg(
    total="count", promedio="mean", minimo="min", maximo="max"
).round(0)
print(resumen.to_string())

print("── Subcategorías con mayor descuento promedio ──")
desc_cat = (
    df_enr[df_enr["descuento_pct"].notna()]
    .groupby("subcategoria")["descuento_pct"]
    .agg(productos="count", descuento_prom="mean")
    .sort_values("descuento_prom", ascending=False)
    .head(10)
    .round(1)
)
print(desc_cat.to_string())

print("── Celulares: precio promedio 5G vs 4G ──")
cel = df_enr[
    (df_enr["subcategoria"] == "celular") &
    (df_enr["precio_venta"].notna())
]
print(cel.groupby("conectividad_5g")["precio_venta"].agg(["count","mean"]).round(0).to_string())

print("── Top 5 marcas por número de productos ──")
print(df_enr["marca_normalizada"].value_counts().head(5).to_string())


── Precio promedio por categoría ──
                  total   promedio    minimo     maximo
categoria                                              
bebes                15   226116.0   34447.0   719940.0
belleza              20   124527.0    5040.0   389898.0
celulares            16  1781416.0  299900.0  5392900.0
cuidado-personal      9    43555.0    1042.0    99999.0
hogar                16   278192.0   26990.0  1389900.0
mercado               3     1338.0     700.0     2520.0
tecnologia           18  1624020.0   69900.0  2849900.0
── Subcategorías con mayor descuento promedio ──
                  productos  descuento_prom
subcategoria                               
monitor                   1            75.0
colchón                   2            65.5
lácteos_y_huevos          3            57.7
televisor                15            56.9
termo                     5            52.4
fragancias               11            52.2
higiene_bebé              1            52.0
otro_bebes     

In [36]:
# ── 14: Verificación final — resumen para Power BI ───────────────────
import os

# Ruta del CSV (se recalcula aquí para que la celda funcione de forma independiente)
CSV_PATH = os.path.join(os.getcwd(), "productos_enriquecidos.csv")

print("═" * 55)
print("  RESUMEN PARA POWER BI")
print("═" * 55)
print(f"  Archivo:              {CSV_PATH}")
print(f"  Total productos:      {len(df_enr):>6}")
print(f"  Con precio_venta:     {df_enr['precio_venta'].notna().sum():>6}")
print(f"  Con descuento:        {int(df_enr['tiene_descuento'].sum()):>6}")
print(f"  Descuento prom:       {df_enr['descuento_pct'].mean():>5.1f}%")
print(f"  Ahorro prom:         ${df_enr['ahorro_cop'].mean():>9,.0f} COP")
print(f"  Categorías:           {df_enr['categoria'].nunique():>6}")
print(f"  Subcategorías:        {df_enr['subcategoria'].nunique():>6}")
print(f"  Marcas únicas:        {df_enr['marca_normalizada'].nunique():>6}")
print(f"  Fecha extracción:     {df_enr['fecha_extraccion'].iloc[0]}")
print("═" * 55)



═══════════════════════════════════════════════════════
  RESUMEN PARA POWER BI
═══════════════════════════════════════════════════════
  Archivo:              c:\Users\bayro\Downloads\Entrega 2 - Corregido\Entrega 2\Entrega 2\productos_enriquecidos.csv
  Total productos:         123
  Con precio_venta:         97
  Con descuento:            81
  Descuento prom:        47.1%
  Ahorro prom:         $  927,080 COP
  Categorías:                7
  Subcategorías:            29
  Marcas únicas:            58
  Fecha extracción:     2026-06-04
═══════════════════════════════════════════════════════
